In [14]:
  # ==============================================================================
  # MODULE 0: DATA PERSISTENCE (LOCAL AUDIT VAULT) - SECURED TO DRIVE
  # ==============================================================================
  import sqlite3
  import os
  from google.colab import drive

  # 1. Montar Google Drive (te pedirá permiso la primera vez)
  drive.mount('/content/drive')

  # 2. Definir una ruta segura en tu Drive
  db_folder = "/content/drive/MyDrive/Trading_Pipeline_Data"
  os.makedirs(db_folder, exist_ok=True)
  db_path = os.path.join(db_folder, "pipeline_audit_vault.db")

  def initialize_vault():
      conn = sqlite3.connect(db_path)
      c = conn.cursor()
      # ... (el resto de tu código de creación de tablas sigue igual) ...
      print(f"✅ Audit Vault initialized securely at: {db_path}")

  initialize_vault()

Mounted at /content/drive
✅ Audit Vault initialized securely at: /content/drive/MyDrive/Trading_Pipeline_Data/pipeline_audit_vault.db


In [2]:
# ==============================================================================
# MODULE 1: TELEGRAM API CONNECTION TEST
# ==============================================================================
import aiohttp
import asyncio
from google.colab import userdata

async def test_telegram_connection():
    try:
        token = userdata.get('TELEGRAM_TOKEN')
        chat_id = userdata.get('TELEGRAM_CHAT_ID')
    except Exception as e:
        print("❌ Error: TELEGRAM_TOKEN or TELEGRAM_CHAT_ID secrets not found in Colab.")
        return

    message = "🤖 SYSTEM TEST: Async Data Pipeline is online and operational."
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    payload = {"chat_id": chat_id, "text": message}

    async with aiohttp.ClientSession() as session:
        async with session.post(url, json=payload) as resp:
            if resp.status == 200:
                print("✅ Connection successful! Check your Telegram.")
            else:
                print(f"❌ HTTP Error {resp.status}")

await test_telegram_connection()

❌ Error: TELEGRAM_TOKEN or TELEGRAM_CHAT_ID secrets not found in Colab.


In [3]:
# ==============================================================================
# MODULE 2: MULTI-TIMEFRAME MACRO VALIDATOR (SMC & FVG)
# ==============================================================================
import asyncio, aiohttp, time, pandas as pd, numpy as np
from datetime import datetime, timezone, timedelta
from tabulate import tabulate
import nest_asyncio; nest_asyncio.apply()

ALL_ASSETS = ['BTC', 'ETH', 'SOL', 'XRP', 'AVAX', 'HYPE', 'xyz:SP500', 'xyz:XYZ100', 'xyz:GOLD', 'xyz:EUR', 'xyz:SILVER', 'xyz:CL']
MACRO_BIAS = {}
LAST_MACRO_UPDATE = None
MACRO_CONCLUSION = {}

TF_PRIORITY = ("1d", "4h", "1h")
LOOKBACK_BARS = 60
SMA_TREND = 50

C_G, C_R, C_Y, C_C, C_rst = "\033[92m", "\033[91m", "\033[93m", "\033[96m", "\033[0m"

def get_utc_time_str():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %I:%M:%S %p UTC")

def colorize(text):
    if text in ["BULL", "BUY"]: return f"{C_G}{text}{C_rst}"
    if text in ["BEAR", "SELL"]: return f"{C_R}{text}{C_rst}"
    if text in ["RANGE"]: return f"{C_Y}{text}{C_rst}"
    return text

def calculate_tf_bias(tf_biases):
    tf1d, tf4h, tf1h = tf_biases
    bull, bear = tf_biases.count("BULL"), tf_biases.count("BEAR")
    if tf1d == "BULL" and tf4h != "BEAR": return "BULL" if tf1h != "BEAR" else "RANGE"
    if tf1d == "BEAR" and tf4h != "BULL": return "BEAR" if tf1h != "BULL" else "RANGE"
    if bull >= 2 and bear == 0: return "BULL"
    if bear >= 2 and bull == 0: return "BEAR"
    return "RANGE"

def calculate_bucket_conclusion(rows, bucket):
    subset = [r[4] for r in rows if ("xyz:" in r[0] if bucket == "TRADFI" else "xyz:" not in r[0])]
    bulls, bears, ranges = subset.count("BULL"), subset.count("BEAR"), subset.count("RANGE")
    if bulls > bears and bulls > ranges: return "BUY"
    if bears > bulls and bears > ranges: return "SELL"
    return "RANGE"

async def fetch_macro_tf(session, coin, interval):
    st_time = int((time.time() - 86400 * 60) * 1000)
    payload = {"type": "candleSnapshot", "req": {"coin": coin, "interval": interval, "startTime": st_time}}
    try:
        async with session.post("https://api.hyperliquid.xyz/info", json=payload, timeout=10) as resp:
            data = await resp.json()
            if not isinstance(data, list) or len(data) < LOOKBACK_BARS: return "N/A"
            df = pd.DataFrame(data)
            for col in ['h', 'l', 'c', 'v']: df[col] = pd.to_numeric(df[col], errors='coerce')
            df = df.dropna(subset=['h', 'l', 'c', 'v'])
            if len(df) < SMA_TREND + 3: return "N/A"

            last_px, sma_trend = df['c'].iloc[-1], df['c'].rolling(SMA_TREND).mean().iloc[-1]
            is_fvg_bull, is_fvg_bear = df['l'].iloc[-1] > df['h'].iloc[-3], df['h'].iloc[-1] < df['l'].iloc[-3]

            if last_px > sma_trend and is_fvg_bull: return "BULL"
            if last_px < sma_trend and is_fvg_bear: return "BEAR"
            if last_px > df['h'].rolling(10).max().iloc[-2] and last_px > sma_trend: return "BULL"
            if last_px < df['l'].rolling(10).min().iloc[-2] and last_px < sma_trend: return "BEAR"
            return "RANGE"
    except: return "N/A"

async def scan_macro_vision():
    global MACRO_BIAS, LAST_MACRO_UPDATE, MACRO_CONCLUSION
    LAST_MACRO_UPDATE = get_utc_time_str()
    async with aiohttp.ClientSession() as session:
        async def scan_coin(coin):
            res_1d, res_4h, res_1h = await asyncio.gather(fetch_macro_tf(session, coin, "1d"), fetch_macro_tf(session, coin, "4h"), fetch_macro_tf(session, coin, "1h"))
            bias = calculate_tf_bias((res_1d, res_4h, res_1h))
            MACRO_BIAS[coin] = bias
            return [coin, colorize(res_1d), colorize(res_4h), colorize(res_1h), bias, colorize(bias)]

        rows = list(await asyncio.gather(*(scan_coin(coin) for coin in ALL_ASSETS)))
        MACRO_CONCLUSION["TRADFI"] = calculate_bucket_conclusion(rows, "TRADFI")
        MACRO_CONCLUSION["CRYPTO"] = calculate_bucket_conclusion(rows, "CRYPTO")

        print(f"{C_C}🕒 LAST MACRO UPDATE: {LAST_MACRO_UPDATE}{C_rst}")
        display_rows = [[r[0], r[1], r[2], r[3], r[5]] for r in rows]
        print(tabulate(display_rows, headers=["ASSET", "1D", "4H", "1H", "FINAL BIAS (SMC)"], tablefmt="simple"))
        print("-" * 90)
        print(f"🌐 MACRO CONCLUSION: TRADFI: {colorize(MACRO_CONCLUSION['TRADFI'])} | CRYPTO: {colorize(MACRO_CONCLUSION['CRYPTO'])}")

await scan_macro_vision()

🕒 LAST MACRO UPDATE: 2026-07-13 04:37:50 PM UTC
ASSET       1D     4H     1H     FINAL BIAS (SMC)
----------  -----  -----  -----  ------------------
BTC         RANGE  BEAR   RANGE  RANGE
ETH         RANGE  RANGE  RANGE  RANGE
SOL         RANGE  BEAR   RANGE  RANGE
XRP         BEAR   BEAR   RANGE  BEAR
AVAX        RANGE  RANGE  RANGE  RANGE
HYPE        BEAR   BEAR   BEAR   BEAR
xyz:SP500   RANGE  BEAR   BEAR   BEAR
xyz:XYZ100  RANGE  BEAR   BEAR   BEAR
xyz:GOLD    BEAR   BEAR   BEAR   BEAR
xyz:EUR     BEAR   BEAR   BEAR   BEAR
xyz:SILVER  BEAR   BEAR   RANGE  BEAR
xyz:CL      RANGE  BULL   RANGE  RANGE
------------------------------------------------------------------------------------------
🌐 MACRO CONCLUSION: TRADFI: SELL | CRYPTO: RANGE


In [5]:
# ==============================================================================
# MODULE 3: ASYNC SIGNAL GENERATOR (AUDIT & MACRO INTEGRATED)
# ==============================================================================
import asyncio, aiohttp, time, json, pandas as pd, numpy as np, sqlite3
from datetime import datetime, timezone, timedelta
from tabulate import tabulate
from IPython.display import clear_output
import nest_asyncio; nest_asyncio.apply()

WATCHLIST = ['BTC', 'ETH', 'SOL', 'HYPE', 'xyz:SP500', 'xyz:XYZ100', 'xyz:GOLD', 'xyz:EUR', 'xyz:SILVER', 'xyz:CL']
HL_API_URL = "https://api.hyperliquid.xyz/info"
MIN_CONFIDENCE = 65
REFRESH_INTERVAL = 5.0
MAX_SIGNALS = 3
MAX_CYCLES = 12
VOL_THRESHOLD = {"CRYPTO": 50000, "TRADFI": 10000}
Z_SCORE_THRESHOLD = 2.0

C_G, C_R, C_Y, C_C, C_rst, C_ELITE = "\033[92m", "\033[91m", "\033[93m", "\033[96m", "\033[0m", "\033[95m"
BEST_SIGNALS = []

def get_local_time_str(): return datetime.now(timezone.utc).strftime("%I:%M:%S %p UTC")
def market_bucket(coin): return "TRADFI" if coin.startswith("xyz:") else "CRYPTO"

def log_signal_to_db(signal, entry, sl, tp, entry_type, direction):
    try:
        conn = sqlite3.connect('pipeline_audit_vault.db')
        c = conn.cursor()
        now = datetime.now(timezone.utc)
        m_bias = globals().get('MACRO_BIAS', {}).get(signal["coin"], "N/A")
        bucket = market_bucket(signal["coin"])
        m_concl = globals().get('MACRO_CONCLUSION', {}).get(bucket, "N/A")
        c.execute('''INSERT INTO signal_registry
                     (date_str, time_str, unix_timestamp, asset, direction,
                      market_price, fvg_close_5m, entry_price, entry_type, stop_loss, take_profit,
                      confidence, zscore, atr_risk, macro_asset_bias, market_conclusion)
                     VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                  (now.strftime("%d/%m/%Y"), now.strftime("%H:%M:%S"), signal["ts"],
                   signal["coin"], direction, signal["price"], signal["close_5m"],
                   entry, entry_type, sl, tp, signal["conf"], signal["zscore"],
                   signal["atr"], m_bias, m_concl))
        conn.commit(); conn.close()
    except: pass

def calculate_risk_levels(price, bias, atr):
    if bias == "BULL": return price * 0.999, (price * 0.999) - (atr * 1.5), (price * 0.999) + (atr * 3.0)
    elif bias == "BEAR": return price * 1.001, (price * 1.001) + (atr * 1.5), (price * 1.001) - (atr * 3.0)
    return price, 0, 0

async def scan_single_coin(session, coin):
    st_time = int((time.time() - 3600 * 4) * 1000)
    payload = {"type": "candleSnapshot", "req": {"coin": coin, "interval": "5m", "startTime": st_time}}
    try:
        async with session.post(HL_API_URL, json=payload, timeout=8) as resp:
            data = await resp.json()
            if not isinstance(data, list) or len(data) < 20: return None
            df = pd.DataFrame(data)
            for col in ['o', 'h', 'l', 'c', 'v']: df[col] = pd.to_numeric(df[col])

            last_p = float(df['c'].iloc[-1])
            ts_p = int(df['t'].iloc[-1] / 1000)
            vol_ok = float(df['v'].iloc[-1]) > VOL_THRESHOLD[market_bucket(coin)]

            df['tr'] = np.maximum(df['h']-df['l'], np.maximum(abs(df['h']-df['c'].shift(1)), abs(df['l']-df['c'].shift(1))))
            atr = df['tr'].rolling(14).mean().iloc[-1]
            bias = "BULL" if df['l'].iloc[-1] > df['h'].iloc[-3] and vol_ok else "BEAR" if df['h'].iloc[-1] < df['l'].iloc[-3] and vol_ok else "RANGE"

            df['delta'] = np.where(df['c'] >= df['o'], df['v'], -df['v'])
            z = (df['delta'].cumsum() - df['delta'].cumsum().rolling(14).mean()) / (df['delta'].cumsum().rolling(14).std() + 1e-9)
            conf = 30 + (35 if bias != "RANGE" else 0) + (25 if abs(z.iloc[-1]) > Z_SCORE_THRESHOLD else 0) + (15 if vol_ok else 0)
            return {"coin": coin, "price": last_p, "close_5m": last_p, "ts": ts_p, "bias": bias, "zscore": float(z.iloc[-1]), "conf": min(conf, 100), "atr": atr}
    except: return None

async def scanner_loop():
    if not globals().get('LAST_MACRO_UPDATE'):
        print(f"{C_R}❌ ERROR: Run Module 2 (Macro) before the Scanner.{C_rst}"); return

    async with aiohttp.ClientSession() as session:
        while True:
            global BEST_SIGNALS
            BEST_SIGNALS, cycle = [], 0
            while len(BEST_SIGNALS) < MAX_SIGNALS and cycle < MAX_CYCLES:
                cycle += 1
                clear_output(wait=True)
                results = await asyncio.gather(*(scan_single_coin(session, coin) for coin in WATCHLIST))
                valid = [r for r in results if r]
                for s in [r for r in valid if r["conf"] >= MIN_CONFIDENCE and r["bias"] != "RANGE"]:
                    if not any(b['coin'] == s['coin'] for b in BEST_SIGNALS): BEST_SIGNALS.append(s)

                print(f"📡 SCANNER ACTIVE | ⏱️ {get_local_time_str()} | Cycle {cycle}/{MAX_CYCLES}")
                print("=" * 115)

                display_table = []
                for r in valid:
                    b_color = C_G if r["bias"] == "BULL" else (C_R if r["bias"] == "BEAR" else C_Y)
                    b_str = f"{b_color}{r['bias']}{C_rst}"
                    z = r["zscore"]
                    z_color = C_C if abs(z) >= Z_SCORE_THRESHOLD else (C_G if z > 0 else C_R)
                    z_str = f"{z_color}{z:+.2f}σ{C_rst}"
                    conf = r["conf"]
                    if conf >= 55: proj = f"{C_Y}🔥 HIGH PROBABILITY{C_rst}"
                    elif conf >= 45: proj = f"{C_C}👀 WARMING UP{C_rst}"
                    else: proj = "💤 COLD"
                    display_table.append([r["coin"], f"${r['price']:.4f}", b_str, z_str, f"{conf}%", f"{r['atr']:.4f}", proj])

                print(tabulate(display_table, headers=["ASSET", "PRICE", "BIAS", "Z-SCORE", "CONF.", "ATR", "PROJECTION"], tablefmt="simple"))
                print("=" * 115)
                print(f"\n{C_C}📖 METRICS GUIDE:{C_rst}")
                print(f"• {C_Y}Z-SCORE:{C_rst} Volume pressure. > +2.0 or < -2.0 indicates strong institutional imbalance.")
                print(f"• {C_Y}CONF:{C_rst} Metric alignment level. Auto-logs at 65%.")
                print(f"• {C_Y}ATR:{C_rst} Current volatility. Defines Stop Loss width to avoid market noise.")
                await asyncio.sleep(REFRESH_INTERVAL)

            if BEST_SIGNALS:
                clear_output(wait=True)
                print(f"{C_G}🎉 SCAN COMPLETE | {len(BEST_SIGNALS)} SIGNALS LOGGED{C_rst}\n" + "="*115)
                for i, s in enumerate(BEST_SIGNALS, 1):
                    entry, sl, tp = calculate_risk_levels(s["price"], s["bias"], s["atr"])
                    entry_t, direct = ("LIMIT" if abs(entry - s["price"]) > 0.001 else "MARKET"), ("LONG" if s["bias"] == "BULL" else "SHORT")
                    log_signal_to_db(s, entry, sl, tp, entry_t, direct)
                    print(f"{i}. {s['coin']} {direct} {entry_t} | 🕒 {get_local_time_str()}")
                    print(f"   🎯 Entry: ${entry:.4f} | SL: ${sl:.4f} | TP: ${tp:.4f}")
                    print(f"   📊 Confidence: {s['conf']}% | Z-Score: {s['zscore']:+.2f}σ | ATR: {s['atr']:.4f}")
                print("="*115 + f"\n{C_C}✅ DATA LOGGED TO LOCAL VAULT.{C_rst}")
                break
            else: print(f"💤 Restarting in {REFRESH_INTERVAL}s..."); await asyncio.sleep(REFRESH_INTERVAL)

try: asyncio.run(scanner_loop())
except KeyboardInterrupt: print("\n⏹️ Scanner stopped.")

📡 SCANNER ACTIVE | ⏱️ 04:38:20 PM UTC | Cycle 2/12
ASSET       PRICE        BIAS    Z-SCORE    CONF.        ATR  PROJECTION
----------  -----------  ------  ---------  -------  -------  -------------------
BTC         $62231.0000  RANGE   -1.70σ     30%      90.6429  💤 COLD
ETH         $1768.1000   RANGE   -1.48σ     30%       2.7143  💤 COLD
SOL         $75.2590     RANGE   -2.48σ     55%       0.1582  🔥 HIGH PROBABILITY
HYPE        $63.1400     RANGE   -1.80σ     30%       0.1719  💤 COLD
xyz:SP500   $7517.0000   RANGE   -1.84σ     30%       4.9643  💤 COLD
xyz:XYZ100  $29290.0000  RANGE   -2.21σ     55%      35.0714  🔥 HIGH PROBABILITY
xyz:GOLD    $3995.5000   RANGE   -2.89σ     55%       3.7286  🔥 HIGH PROBABILITY
xyz:EUR     $1.1387      BEAR    -1.97σ     80%       0.0002  🔥 HIGH PROBABILITY
xyz:SILVER  $57.5520     BEAR    -1.67σ     80%       0.1213  🔥 HIGH PROBABILITY
xyz:CL      $75.0350     RANGE   +0.83σ     45%       0.1714  👀 WARMING UP

📖 METRICS GUIDE:
• Z-SCORE: Volume pr

In [6]:
# ==============================================================================
# MODULE 3.5: TRADE PREPARATION TERMINAL (AUTHOR DETECTION)
# ==============================================================================
import re
from IPython.display import display, clear_output
import ipywidgets as widgets
import nest_asyncio; nest_asyncio.apply()

C_G, C_R, C_Y, C_C, C_rst = "\033[92m", "\033[91m", "\033[93m", "\033[96m", "\033[0m"

if 'PIPELINE_TRADE_CONFIG' not in globals():
    PIPELINE_TRADE_CONFIG = {}

def build_terminal():
    print(f"{C_C}🖥️ TRADE PREPARATION TERMINAL{C_rst}")
    alert_text = widgets.Textarea(placeholder='Paste Scanner alert or write manually...', layout=widgets.Layout(width='600px', height='120px'))

    author_input = widgets.Dropdown(options=['USER', 'SYSTEM'], value='USER', description='Author:', layout=widgets.Layout(width='180px'))
    coin_input = widgets.Text(description='Asset:', value='', layout=widgets.Layout(width='180px'))
    type_input = widgets.Dropdown(options=['LONG', 'SHORT'], value='LONG', description='Type:', layout=widgets.Layout(width='150px'))
    entry_type_input = widgets.Dropdown(options=['LIMIT', 'MARKET'], value='LIMIT', description='Mode:', layout=widgets.Layout(width='150px'))
    entry_input = widgets.FloatText(description='Price:', value=0.0)
    sl_input = widgets.FloatText(description='SL:', value=0.0)
    tp_input = widgets.FloatText(description='TP:', value=0.0)

    btn_parse = widgets.Button(description='🔍 1. Parse Data', button_style='info', layout=widgets.Layout(width='195px'))
    btn_save = widgets.Button(description='💾 2. Confirm', button_style='success', layout=widgets.Layout(width='195px'))
    btn_clear = widgets.Button(description='🧹 Clear', button_style='warning', layout=widgets.Layout(width='195px'))

    output_area = widgets.Output()

    def on_clear(b):
        alert_text.value = ''
        coin_input.value = ''
        type_input.value = 'LONG'
        entry_type_input.value = 'LIMIT'
        entry_input.value = 0.0
        sl_input.value = 0.0
        tp_input.value = 0.0
        author_input.value = 'USER'
        with output_area:
            clear_output()
            print(f"{C_Y}🧹 Fields reset.{C_rst}")

    def on_parse(b):
        with output_area:
            clear_output()
            text = alert_text.value.upper()
            if not text.strip(): return

            # Author Detection (Falls back to SYSTEM if it contains scanner keywords)
            if "⭐" in alert_text.value or "CONFIDENCE" in text or "CONF." in text:
                author_input.value = "SYSTEM"
                print(f"{C_C}🤖 SIGNAL DETECTED: Origin Async Scanner{C_rst}")
            else:
                author_input.value = "USER"
                print(f"{C_Y}👤 MANUAL ENTRY: Origin User{C_rst}")

            match_main = re.search(r'(?:(\w+):)?(\b\w+\b)\s+\b(LONG|SHORT)\b', text)
            if match_main:
                prefix, asset, direction = match_main.groups()
                type_input.value = direction
                coin_input.value = f"{prefix.lower()}:{asset}" if prefix else asset

            sl_m = re.search(r'(?:SL|STOP|LOSS)[:\s]*\$?([0-9.]+)', text)
            tp_m = re.search(r'(?:TP|TAKE|PROFIT)[:\s]*\$?([0-9.]+)', text)
            entry_m = re.search(r'(?:ENTRY|🎯)[:\s]*\$?([0-9.]+)', text)

            if sl_m: sl_input.value = float(sl_m.group(1))
            if tp_m: tp_input.value = float(tp_m.group(1))
            if entry_m: entry_input.value = float(entry_m.group(1))

            if "MARKET" in text:
                entry_type_input.value = "MARKET"
                if not entry_m: entry_input.value = 0.0
            else:
                entry_type_input.value = "LIMIT"
                if not entry_m:
                    nums = re.findall(r'\b\$?([0-9.]+)\b', text)
                    for n in nums:
                        val = float(n)
                        if val not in [sl_input.value, tp_input.value] and val > 10:
                            entry_input.value = val; break

    def on_confirm(b):
        global PIPELINE_TRADE_CONFIG
        with output_area:
            clear_output()
            PIPELINE_TRADE_CONFIG = {
                'coin': coin_input.value, 'type': type_input.value, 'entry_type': entry_type_input.value,
                'entry': float(entry_input.value), 'sl': float(sl_input.value), 'tp': float(tp_input.value),
                'author': author_input.value
            }
            print(f"{C_G}✅ CONFIGURATION LOCKED. AUTHOR: {PIPELINE_TRADE_CONFIG['author']}{C_rst}")
            print(f"🚀 Monitor ready for {PIPELINE_TRADE_CONFIG['coin']} at ${PIPELINE_TRADE_CONFIG['entry']:.4f}")

    btn_parse.on_click(on_parse)
    btn_save.on_click(on_confirm)
    btn_clear.on_click(on_clear)

    display(alert_text,
            widgets.HBox([btn_parse, btn_save, btn_clear]),
            widgets.HBox([coin_input, type_input, entry_type_input, author_input]),
            widgets.HBox([entry_input, sl_input, tp_input]),
            output_area)

build_terminal()

🖥️ TRADE PREPARATION TERMINAL


Textarea(value='', layout=Layout(height='120px', width='600px'), placeholder='Paste Scanner alert or write man…

Output()

In [7]:
# ==============================================================================
# MODULE 4: EXECUTION MONITOR (WEBSOCKET & KILL SWITCH)
# ==============================================================================
import nest_asyncio; nest_asyncio.apply()
import asyncio, aiohttp, time, json, sqlite3
from datetime import datetime, timezone, timedelta
from IPython.display import clear_output
from tabulate import tabulate

C_G, C_R, C_Y, C_C, C_rst = "\033[92m", "\033[91m", "\033[93m", "\033[96m", "\033[0m"

class ExecutionMonitor:
    def __init__(self, config):
        self.config = config
        raw_coin = config.get('coin', '').strip().upper()
        tradfi_list = ['GOLD', 'SILVER', 'EUR', 'SP500', 'XYZ100', 'CL', 'WTIOIL']
        self.coin = f"xyz:{raw_coin}" if raw_coin in tradfi_list else raw_coin

        self.type = config.get('type', 'LONG').upper()
        self.entry_type = config.get('entry_type', 'MARKET').upper()
        self.entry_target = float(config.get('entry', 0.0))
        self.sl = float(config.get('sl', 0.0))
        self.tp = float(config.get('tp', 0.0))
        self.author = config.get('author', 'USER')

        self.last_px, self.active, self.filled = 0.0, True, False
        self.exit_reason, self.exit_price = None, 0.0
        self.scan_start_time = time.time()
        self.position_start_time = None
        self.last_tick = time.time()

    async def fetch_price_rest(self):
        try:
            async with aiohttp.ClientSession() as session:
                for ticker in [self.coin, self.coin.replace("xyz:","")]:
                    payload = {"type": "l2Book", "coin": ticker}
                    async with session.post("https://api.hyperliquid.xyz/info", json=payload) as resp:
                        data = await resp.json()
                        levels = data.get('levels', [])
                        if levels and len(levels[0]) > 0:
                            self.last_px = float(levels[0][0]['px'])
                            return True
        except: pass
        return False

    def log_final_trade(self, status_override=None):
        try:
            conn = sqlite3.connect('pipeline_audit_vault.db')
            c = conn.cursor()
            try: c.execute("ALTER TABLE trade_history ADD COLUMN duration_sec INTEGER")
            except: pass

            date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
            pnl, final_author = 0.0, self.author
            exit_px = self.exit_price if self.exit_price > 0 else self.last_px

            duration = 0
            if self.position_start_time:
                duration = int(time.time() - self.position_start_time)

            if not self.filled: final_author = f"{self.author} (CANCELLED - TIMEOUT)"
            else: pnl = ((exit_px/self.entry_target)-1)*100 if self.type=="LONG" else ((self.entry_target/exit_px)-1)*100

            if status_override: final_author = f"{self.author} ({status_override})"

            c.execute("""INSERT INTO trade_history
                         (date, asset, direction, entry_price, exit_price, final_pnl, author, duration_sec)
                         VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                      (date_str, self.coin, self.type, self.entry_target, exit_px, round(pnl, 4), final_author, duration))
            conn.commit(); conn.close()
            return True
        except Exception as e:
            print(f"Error logging to Vault: {e}")
            return False

    def check_exits(self):
        if not self.filled or self.last_px == 0: return
        if self.type == "LONG":
            if self.sl > 0 and self.last_px <= self.sl: self.exit_reason, self.exit_price, self.active = "STOP LOSS", self.sl, False
            elif self.tp > 0 and self.last_px >= self.tp: self.exit_reason, self.exit_price, self.active = "TAKE PROFIT", self.tp, False
        elif self.type == "SHORT":
            if self.sl > 0 and self.last_px >= self.sl: self.exit_reason, self.exit_price, self.active = "STOP LOSS", self.sl, False
            elif self.tp > 0 and self.last_px <= self.tp: self.exit_reason, self.exit_price, self.active = "TAKE PROFIT", self.tp, False

    async def ws_listener(self, ws):
        clean = self.coin.replace("xyz:","")
        while self.active:
            msg = await ws.receive()
            try:
                data = json.loads(msg.data)
                d = data.get('data', {})
                coin_msg = d.get('coin') if isinstance(d, dict) else d[0].get('coin') if isinstance(d, list) else ""
                if coin_msg in [self.coin, clean]:
                    self.last_tick = time.time()
                    if data.get('channel') == 'l2Book' and d.get('levels'): self.last_px = float(d['levels'][0][0]['px'])
                    elif data.get('channel') == 'trades' and isinstance(d, list): self.last_px = float(d[0]['px'])

                    if not self.filled:
                        is_long_entry = (self.type == "LONG" and self.last_px <= self.entry_target)
                        is_short_entry = (self.type == "SHORT" and self.last_px >= self.entry_target)

                        if self.entry_type == "MARKET" or is_long_entry or is_short_entry:
                            self.filled = True
                            self.position_start_time = time.time()
                            if self.entry_type == "MARKET": self.entry_target = self.last_px

                    self.check_exits()
            except: continue

    async def render_loop(self):
        while self.active:
            clear_output(wait=True)
            print(f"{C_C}📡 EXECUTION MONITOR (AUTHOR: {self.author}){C_rst}")

            if self.last_px == 0:
                print(f"{C_Y}🔄 ESTABLISHING CONNECTION (FORCING READ WITH {self.coin})...{C_rst}")
                await self.fetch_price_rest()

            elif not self.filled:
                dist = abs(self.last_px - self.entry_target)
                elap = int(time.time() - self.scan_start_time)
                tick_str = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')

                print(f"{C_Y}⏳ AWAITING ENTRY ({self.entry_type}){C_rst}")
                print("-" * 60)
                table = [
                    ["ASSET", f"{C_C}{self.coin}{C_rst}"],
                    ["DIRECTION", f"{C_G if self.type == 'LONG' else C_R}{self.type}{C_rst}"],
                    ["TARGET", f"{C_C}${self.entry_target:.4f}{C_rst}"],
                    ["STOP LOSS", f"{C_R}${self.sl:.4f}{C_rst}"],
                    ["TAKE PROFIT", f"{C_G}${self.tp:.4f}{C_rst}"],
                    ["WAIT TIME", f"{elap // 60:02d}:{elap % 60:02d}"],
                    ["LAST TICK", tick_str],
                    ["MARKET", f"${self.last_px:.4f}"],
                    ["DISTANCE", f"{C_Y}${dist:.6f}{C_rst}"]
                ]
                print(tabulate(table, tablefmt="plain"))
                print("-" * 60)
            else:
                pnl = ((self.last_px/self.entry_target)-1)*100 if self.type=="LONG" else ((self.entry_target/self.last_px)-1)*100
                trade_elap = int(time.time() - self.position_start_time)

                print(f"{C_G}🚀 POSITION OPEN (IN MARKET){C_rst}")
                print(tabulate([
                    ["ASSET", self.coin],
                    ["DIR", self.type],
                    ["TIME IN POS.", f"{C_Y}{trade_elap // 60:02d}:{trade_elap % 60:02d}{C_rst}"],
                    ["PnL %", f"{C_G if pnl >=0 else C_R}{pnl:+.2f}%{C_rst}"],
                    ["ENTRY", f"${self.entry_target:.4f}"],
                    ["MARKET", f"${self.last_px:.4f}"],
                    ["SL / TP", f"{C_R}${self.sl:.4f}{C_rst} / {C_G}${self.tp:.4f}{C_rst}"]
                ], tablefmt="plain"))
            await asyncio.sleep(1.5)
        self.log_final_trade()

    async def run(self):
        await self.fetch_price_rest()
        async with aiohttp.ClientSession() as session:
            async with session.ws_connect("wss://api.hyperliquid.xyz/ws") as ws:
                await ws.send_json({"method": "subscribe", "subscription": {"type": "trades", "coin": self.coin}})
                await ws.send_json({"method": "subscribe", "subscription": {"type": "l2Book", "coin": self.coin}})
                await asyncio.gather(self.ws_listener(ws), self.render_loop())

inst_mon = None
try:
    config = globals().get('PIPELINE_TRADE_CONFIG', {})
    if config: inst_mon = ExecutionMonitor(config); asyncio.run(inst_mon.run())
except KeyboardInterrupt:
    if inst_mon:
        inst_mon.active = False
        status = "TIMEOUT - CANCELLED" if not inst_mon.filled else "MANUAL CLOSE"
        inst_mon.log_final_trade(status_override=status)

In [8]:
# ==============================================================================
# MODULE 4.2: GLOBAL WEBSOCKET DIAGNOSTIC (NETWORK ISOLATION)
# ==============================================================================
import asyncio, aiohttp, json
import nest_asyncio; nest_asyncio.apply()
from tabulate import tabulate

C_G, C_R, C_Y, C_C, C_rst = "\033[92m", "\033[91m", "\033[93m", "\033[96m", "\033[0m"

async def global_ws_diagnostic():
    print(f"{C_C}🔍 INITIATING WEBSOCKET INFRASTRUCTURE DIAGNOSTIC...{C_rst}\n")

    assets_to_test = {
        'BTC': ['BTC'], 'ETH': ['ETH'], 'SOL': ['SOL'], 'HYPE': ['HYPE'],
        'SP500': ['xyz:SP500', 'SP500'], 'XYZ100': ['xyz:XYZ100', 'XYZ100'],
        'GOLD': ['xyz:GOLD', 'GOLD'], 'EUR': ['xyz:EUR', 'EUR'],
        'SILVER': ['xyz:SILVER', 'SILVER'], 'CL': ['xyz:CL', 'CL']
    }

    results = []

    async with aiohttp.ClientSession() as session:
        for base_asset, variants in assets_to_test.items():
            asset_status = f"{C_R}❌ REJECTED{C_rst}"
            working_variant = "NONE"

            for variant in variants:
                try:
                    async with session.ws_connect("wss://api.hyperliquid.xyz/ws") as ws:
                        sub = {"method": "subscribe", "subscription": {"type": "l2Book", "coin": variant}}
                        await ws.send_json(sub)

                        for _ in range(4):
                            msg = await asyncio.wait_for(ws.receive(), timeout=2.0)
                            if msg.type == aiohttp.WSMsgType.TEXT:
                                data = json.loads(msg.data)
                                if data.get("channel") == "l2Book" and data["data"]["coin"] == variant:
                                    asset_status = f"{C_G}✅ CONNECTED{C_rst}"
                                    working_variant = variant
                                    break
                            elif msg.type in (aiohttp.WSMsgType.CLOSED, aiohttp.WSMsgType.ERROR):
                                break
                except Exception:
                    pass

                if "CONNECTED" in asset_status:
                    break

            results.append([base_asset, working_variant, asset_status])
            print(f"[{'✔️' if 'CONNECTED' in asset_status else '❌'}] Evaluating {base_asset}... Accepted: {working_variant}")

    print("\n" + "="*65)
    print(f"{C_Y}📊 FINAL WEBSOCKET DIAGNOSTIC REPORT{C_rst}")
    print("="*65)
    print(tabulate(results, headers=["BASE ASSET", "REQUIRED TICKER (WS)", "STATUS"], tablefmt="simple"))
    print("="*65)

try:
    asyncio.run(global_ws_diagnostic())
except KeyboardInterrupt:
    print("\n⏹️ Diagnostic stopped.")

🔍 INITIATING WEBSOCKET INFRASTRUCTURE DIAGNOSTIC...

[✔️] Evaluating BTC... Accepted: BTC
[✔️] Evaluating ETH... Accepted: ETH
[✔️] Evaluating SOL... Accepted: SOL
[✔️] Evaluating HYPE... Accepted: HYPE
[✔️] Evaluating SP500... Accepted: xyz:SP500
[✔️] Evaluating XYZ100... Accepted: xyz:XYZ100
[✔️] Evaluating GOLD... Accepted: xyz:GOLD
[✔️] Evaluating EUR... Accepted: xyz:EUR
[✔️] Evaluating SILVER... Accepted: xyz:SILVER
[✔️] Evaluating CL... Accepted: xyz:CL

📊 FINAL WEBSOCKET DIAGNOSTIC REPORT
BASE ASSET    REQUIRED TICKER (WS)    STATUS
------------  ----------------------  ------------
BTC           BTC                     ✅ CONNECTED
ETH           ETH                     ✅ CONNECTED
SOL           SOL                     ✅ CONNECTED
HYPE          HYPE                    ✅ CONNECTED
SP500         xyz:SP500               ✅ CONNECTED
XYZ100        xyz:XYZ100              ✅ CONNECTED
GOLD          xyz:GOLD                ✅ CONNECTED
EUR           xyz:EUR                 ✅ CONNECTED
SI

In [9]:
# ==============================================================================
# MODULE 5: SIGNAL AUDIT EXPORTER (EXCEL)
# ==============================================================================
import pandas as pd, sqlite3, ipywidgets as widgets
from google.colab import files
from IPython.display import display

def export_audit_to_excel(b):
    date_filter = f"{day_in.value:02d}/{month_in.value:02d}/{year_in.value}"
    conn = sqlite3.connect('pipeline_audit_vault.db')
    df = pd.read_sql_query("SELECT * FROM signal_registry WHERE date_str = ?", conn, params=(date_filter,))
    conn.close()

    if df.empty:
        print(f"❌ No data found for {date_filter}")
    else:
        df.columns = ['ID', 'Date', 'Time', 'TS_Unix', 'Asset', 'Direction', 'Market_Px', 'FVG_Close_5m',
                      'Entry_Px', 'Entry_Mode', 'SL', 'TP', 'Confidence %', 'Z-Score', 'ATR',
                      'Macro_Bias', 'Market_Conclusion', 'Max_15m (Wick)', 'Min_15m (Wick)', 'Px_4h (Trend)']
        file_name = f"Audit_Full_{date_filter.replace('/','-')}.xlsx"
        df.to_excel(file_name, index=False)
        files.download(file_name)
        print(f"✅ Institutional report downloaded: {file_name}")

day_in = widgets.IntText(value=15, description='Day:', layout={'width': '120px'})
month_in = widgets.IntText(value=4, description='Month:', layout={'width': '120px'})
year_in = widgets.IntText(value=2026, description='Year:', layout={'width': '150px'})
btn_export = widgets.Button(description='Download Audit', button_style='success')
btn_export.on_click(export_audit_to_excel)
display(widgets.HBox([day_in, month_in, year_in, btn_export]))

In [11]:
# ==============================================================================
# MODULE 6: POST-MARKET AUDIT (WICK & TREND VALIDATION)
# ==============================================================================
import asyncio, aiohttp, sqlite3, pandas as pd, time
from IPython.display import clear_output

async def run_post_market_audit():
    print("🔍 Initiating precision audit on the local database...")
    conn = sqlite3.connect('pipeline_audit_vault.db')
    c = conn.cursor()

    c.execute("SELECT id, asset, unix_timestamp FROM signal_registry WHERE max_15m IS NULL")
    pending = c.fetchall()

    if not pending:
        print("✅ All signals are fully audited.")
        return

    async with aiohttp.ClientSession() as session:
        for s_id, asset, ts in pending:
            print(f"📊 Auditing {asset} (ID: {s_id})...")

            # 1. Wick check (15 min on 1m candles)
            p_wick = {"type": "candleSnapshot", "req": {"coin": asset, "interval": "1m", "startTime": ts * 1000}}
            async with session.post("https://api.hyperliquid.xyz/info", json=p_wick) as r:
                d_w = await r.json()
                if isinstance(d_w, list) and len(d_w) >= 15:
                    df_w = pd.DataFrame(d_w[:15])
                    max_15, min_15 = pd.to_numeric(df_w['h']).max(), pd.to_numeric(df_w['l']).min()
                    c.execute("UPDATE signal_registry SET max_15m = ?, min_15m = ? WHERE id = ?", (max_15, min_15, s_id))

            # 2. 4h Trend validation
            ts_4h = (ts + (3600 * 4)) * 1000
            p_4h = {"type": "candleSnapshot", "req": {"coin": asset, "interval": "1h", "startTime": ts_4h}}
            async with session.post("https://api.hyperliquid.xyz/info", json=p_4h) as r:
                d_4 = await r.json()
                if isinstance(d_4, list) and len(d_4) > 0:
                    px_4h = float(d_4[0]['c'])
                    c.execute("UPDATE signal_registry SET px_4h = ? WHERE id = ?", (px_4h, s_id))

            conn.commit()
            await asyncio.sleep(0.5) # Rate limit protection

    conn.close()
    print(f"🏁 Audit complete. {len(pending)} records updated with reality data.")

await run_post_market_audit()

🔍 Initiating precision audit on the local database...
✅ All signals are fully audited.


In [12]:
# ==============================================================================
# MODULE 7: FULL AUDIT SWEEP & DIRECT DOWNLOAD
# ==============================================================================
import asyncio, aiohttp, sqlite3, pandas as pd, time
from google.colab import files

async def execute_audit_and_download():
    print("🔍 Initiating precision sweep and report preparation...")
    conn = sqlite3.connect('pipeline_audit_vault.db')
    c = conn.cursor()

    # 1. Sweep missing data (Logic from Module 6)
    c.execute("SELECT id, asset, unix_timestamp FROM signal_registry WHERE max_15m IS NULL")
    pending = c.fetchall()

    if pending:
        async with aiohttp.ClientSession() as session:
            for s_id, asset, ts in pending:
                p_wick = {"type": "candleSnapshot", "req": {"coin": asset, "interval": "1m", "startTime": ts * 1000}}
                async with session.post("https://api.hyperliquid.xyz/info", json=p_wick) as r:
                    d_w = await r.json()
                    if isinstance(d_w, list) and len(d_w) >= 15:
                        df_w = pd.DataFrame(d_w[:15])
                        max_15, min_15 = pd.to_numeric(df_w['h']).max(), pd.to_numeric(df_w['l']).min()
                        c.execute("UPDATE signal_registry SET max_15m = ?, min_15m = ? WHERE id = ?", (max_15, min_15, s_id))

                ts_4h = (ts + (3600 * 4)) * 1000
                p_4h = {"type": "candleSnapshot", "req": {"coin": asset, "interval": "1h", "startTime": ts_4h}}
                async with session.post("https://api.hyperliquid.xyz/info", json=p_4h) as r:
                    d_4 = await r.json()
                    if isinstance(d_4, list) and len(d_4) > 0:
                        px_4h = float(d_4[0]['c'])
                        c.execute("UPDATE signal_registry SET px_4h = ? WHERE id = ?", (px_4h, s_id))
                conn.commit()
                await asyncio.sleep(0.3)
        print(f"✅ {len(pending)} records updated with reality data.")

    # 2. Direct Export (Logic from Module 5)
    df = pd.read_sql_query("SELECT * FROM signal_registry", conn)
    conn.close()

    file_name = "Institutional_Audit_Final.xlsx"
    df.to_excel(file_name, index=False)
    files.download(file_name)
    print(f"🚀 Report downloaded successfully: {file_name}")

await execute_audit_and_download()

🔍 Initiating precision sweep and report preparation...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Report downloaded successfully: Institutional_Audit_Final.xlsx


In [13]:
# ==============================================================================
# 🩺 SONDA HIP-3 V6.3 | STRESS TEST (BACKEND ALIGNED)
# ==============================================================================
import asyncio, aiohttp, time, json
import pandas as pd
import nest_asyncio; nest_asyncio.apply()

HL_API_URL = "https://api.hyperliquid.xyz/info"

TEST_WATCHLIST = [
    'xyz:SP500', 'xyz:XYZ100', 'xyz:GOLD', 'xyz:EUR',
    'xyz:SILVER', 'xyz:CL'
]

C_G, C_R, C_Y, C_C, C_rst = "\033[92m", "\033[91m", "\033[93m", "\033[96m", "\033[0m"

async def probe_expanded():
    print(f"{C_C}🚀 SONDA HIP-3 V6.3 EXTENSIVA...{C_rst}\n")
    async with aiohttp.ClientSession() as session:
        for coin in TEST_WATCHLIST:
            print(f"➤ {C_Y}{coin}{C_rst}")

            # 1. VELAS
            st_time = int((time.time() - 86400 * 3) * 1000)
            payload_c = {"type": "candleSnapshot", "req": {"coin": coin, "interval": "5m", "startTime": st_time}}
            try:
                async with session.post(HL_API_URL, json=payload_c, timeout=12) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        if isinstance(data, list) and len(data) > 0:
                            print(f"  {C_G}✅ VELAS OK ({len(data)}){C_rst} C:${float(data[-1]['c']):.2f}")
                        else: print(f"  {C_Y}⚠️ VELAS VACÍAS{C_rst}")
                    else: print(f"  {C_R}❌ VELAS HTTP {resp.status}{C_rst}")
            except Exception as e: print(f"  {C_R}❌ VELAS ERR:{C_rst} {str(e)[:30]}")

            # 2. L2BOOK
            payload_l2 = {"type": "l2Book", "coin": coin}
            try:
                async with session.post(HL_API_URL, json=payload_l2, timeout=12) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        levels = data.get('levels', [[], []])
                        if levels and len(levels[0]) > 0:
                            mid = (float(levels[0][0]['px']) + float(levels[1][0]['px'])) / 2
                            print(f"  {C_G}✅ L2 OK{C_rst} Mid:${mid:.2f}")
                        else: print(f"  {C_Y}⚠️ L2 VACÍO{C_rst}")
                    else: print(f"  {C_R}❌ L2 HTTP {resp.status}{C_rst}")
            except Exception as e: print(f"  {C_R}❌ L2 ERR:{C_rst} {str(e)[:30]}")
            print("-" * 50)

try: asyncio.run(probe_expanded())
except KeyboardInterrupt: print(f"\n{C_Y}Sonda detenida.{C_rst}")

🚀 SONDA HIP-3 V6.3 EXTENSIVA...

➤ xyz:SP500
  ✅ VELAS OK (865) C:$7528.10
  ✅ L2 OK Mid:$7528.15
--------------------------------------------------
➤ xyz:XYZ100
  ✅ VELAS OK (865) C:$29353.00
  ✅ L2 OK Mid:$29351.00
--------------------------------------------------
➤ xyz:GOLD
  ✅ VELAS OK (865) C:$4002.60
  ✅ L2 OK Mid:$4002.55
--------------------------------------------------
➤ xyz:EUR
  ✅ VELAS OK (863) C:$1.14
  ✅ L2 OK Mid:$1.14
--------------------------------------------------
➤ xyz:SILVER
  ✅ VELAS OK (865) C:$57.66
  ✅ L2 OK Mid:$57.65
--------------------------------------------------
➤ xyz:CL
  ✅ VELAS OK (865) C:$75.00
  ✅ L2 OK Mid:$75.00
--------------------------------------------------


# 🔄 Asynchronous Market Data Pipeline (Personal Learning Project)

### ⚠️ Project Context
This repository contains a Jupyter Notebook documenting my personal journey to master **advanced asynchronous Python** and **real-time data ingestion**.

My goal was **not** to build a commercial trading system or a quantitative finance product. Instead, I challenged myself to use AI-assisted development ("Vibe Coding") to engineer a resilient, non-blocking data pipeline from scratch that connects to external WebSockets, processes streaming JSON, and logs it safely into a local database for post-market auditing.

---

## 🛠️ Technical Concepts Explored & Implemented

- **Asynchronous Concurrency (`asyncio` & `aiohttp`):** Learned how to maintain persistent WebSocket connections to stream real-time market data (L2 Order Books, Candlesticks) alongside REST API calls without blocking the main execution thread.
- **Vectorized Data Processing (`pandas` & `numpy`):** Applied fast, vectorized mathematical operations to streaming data to calculate volatility metrics (ATR) and structural zones in real-time.
- **Local Data Persistence (`sqlite3`):** Designed a relational schema ("The Vault") to log streaming data and generated signals, ensuring zero data loss during network spikes and enabling historical querying.
- **Resilient Error Handling:** Implemented network diagnostic tools, graceful `KeyboardInterrupt` exits, and strict JSON parsing fallbacks to prevent pipeline crashes.
- **AI-Assisted Development:** Leveraged LLMs as a pair-programmer to understand the complexities of the `asyncio` event loop, debug WebSocket disconnection errors, and structure the database schema efficiently.

---

## 📂 Notebook Architecture (Modular Design)

1. **Module 0: Audit Vault Initialization** – Sets up the local `sqlite3` database for persistent logging of signals and executed actions.
2. **Module 1 & 8: Network & API Diagnostics** – Pre-flight checks to validate WebSocket connectivity and API health before initiating heavy data streams.
3. **Module 2: Multi-Timeframe Macro Validator** – Fetches and processes historical REST data to establish baseline structural bias (non-blocking).
4. **Module 3: Async Signal Scanner** – The core loop. Concurrently scans multiple assets via WebSocket/REST, calculates metrics, and logs high-probability structural setups to the database.
5. **Module 4: Execution Monitor** – A real-time state machine that tracks active conditions, manages kill-switches, and logs final outcomes with duration metrics.
6. **Module 5 & 6: Post-Market Audit** – Retroactively fetches missing high-resolution data (e.g., 1m wicks, 4h trends) to validate the accuracy of the initial signal against market reality, exporting clean reports to Excel.

---

## 💡 Key Takeaway

This project proved my ability to take complex, abstract technical concepts (like async concurrency and WebSocket state management), break them down using business logic, and use AI to execute and deploy a functional, resilient data system.

*Note: This is a personal engineering project focused on data pipeline architecture and system reliability. It is not financial advice nor a commercial financial product.*